# Create Character-Set Diagrams of the Dreadnoughts 

In [531]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

### Load the Character Sets

In [532]:
import re
from colors_map import *

def loadCharacterSet(charset_lines, ref_offset=0):
    character_set = {}
    charset_data = []
    char_ref = None
    for l in charset_lines:
        if "CHARACTER" in l:
            if charset_data:
                character_set[char_ref] = charset_data
            charset_data = []
            ref = int(l[61:63],16) + ref_offset
            char_ref = f"{ref:0{2}x}".upper()
             
        m = re.findall(r"[0-1]{8}",l)
        if not m:
            continue
        line_bits = m[0].strip()
        charset_data += [line_bits]
    character_set[char_ref] = charset_data
    return character_set

charsets_file = "uridium/src/charset.asm"
input_file = open(charsets_file,'r')
charset_lines = input_file.readlines()
base_charset_lines = charset_lines[:1292]
base_character_set = loadCharacterSet(base_charset_lines)

logo_charset_lines = charset_lines[1440:1647]
logo_character_set = loadCharacterSet(logo_charset_lines)

## Generate PNG images of the Base Character Set

In [533]:
from PIL import Image, ImageColor
CHARACTER_COLS = 8
CHARACTER_ROWS = 8

def paintRawMonoCharacter(character_set, character, color="purple", verticalExpand=False):
    CHARACTER_COLS = 8
    CHARACTER_ROWS = 8

    if character not in character_set:
        print(character)
        return
    
    image_width = CHARACTER_COLS
    image_height = CHARACTER_ROWS
    img = Image.new( 'RGBA', (image_width, image_height))
    pixels = img.load()

    bit_array = character_set[character]
    for y, l in enumerate(bit_array):
        for x,bit in enumerate(l):
            if bit == "0":
                continue
            pixel_color = ImageColor.getrgb(color)
            pixels[x,y] = pixel_color
    return img

In [534]:
char_colors = {
"c64_white": "#ffffff",     
"c64_green": "#56ac4d",     
"c64_yellow":  "#edf171",   
"c64_ltred": "#c46c71",  
"c64_ltgreen": "#a9ff9f",
"c64_ltblue":  "#706deb",
"c64_red": "#813338",       
"c64_green": "#56ac4d",     
"c64_blue":  "#2e2c9b",     
"c64_purple": "#8e3c97",    
}

In [535]:
!mkdir -p text_character_set_diagrams/base_character_sets

In [536]:
from PIL import ImageDraw, ImageFont

for color_name, color in char_colors.items():
    for character_name in base_character_set:
        img = paintRawMonoCharacter(base_character_set, character_name, color=color)
        img = img.resize((img.width * 15, img.height * 15), Image.NEAREST)
        framed_img = img.copy()
        draw = ImageDraw.Draw(framed_img)
        draw.rectangle((0, 0, img.width-1, img.height-1), fill=None, outline="black")

        if character_name:
            img.save(f"text_character_set_diagrams/base_character_sets/{color_name}_{character_name}.png")
            framed_img.save(f"text_character_set_diagrams/base_character_sets/{color_name}_{character_name}_framed.png")


## Create PNG Tables of the Base Characterset

In [537]:
from PIL import Image, ImageColor
CHARACTER_COLS = 8
CHARACTER_ROWS = 8
CELL_WIDTH = 40
CELL_HEIGHT = 40

def paintCharacterDiagram(character_set, character, color="purple"):
    image_width = CELL_WIDTH*CHARACTER_COLS
    image_height = CELL_HEIGHT*CHARACTER_ROWS
    img = Image.new( 'RGBA', (image_width+1, image_height+1))
    draw = ImageDraw.Draw(img)

    fnt = ImageFont.truetype("RobotoMono-Light.ttf", 40)
    bit_array = character_set[character]
    # Remember that each bitpair in the bit_array is duplicated.
    # e.g. 01 appears as 01,01 so that we can treat each element as
    # a single bit.
    for y, l in enumerate(bit_array):
        for x,bit in enumerate(l):
            background_color = color if bit == '1' else "white"
            pixel_color = ImageColor.getrgb(background_color) 
            X = x * CELL_WIDTH
            Y = y * CELL_HEIGHT
            draw.rectangle((X, Y, X+CELL_WIDTH, Y+CELL_HEIGHT), 
                           fill=pixel_color, outline="black")
            text_color = "black" if background_color == "white" else "white"
            draw.text((X+10, Y-8), bit, font=fnt, fill=text_color)
    return img


In [538]:
!mkdir -p text_character_set_diagrams/base_character_set_tables

In [539]:
for color_name, color in char_colors.items():
    for character_name in base_character_set:
        img = paintCharacterDiagram(base_character_set, character_name, color=color)
        if character_name:
            img.save(f"text_character_set_diagrams/base_character_set_tables/{color_name}_{character_name}.png")

In [540]:
byte_literals = """.BYTE
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
.BYTE 
"""

def generateCharacterDiagram(character_image, character_name, character_bytes):

    img = Image.new('RGBA', (540,350))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"CHAR/{character_name}"
    label_fnt_size = 29
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (5, (img.height - (label.height + 20))))

    # Sprite byte literals
    label_text = byte_literals
    label_fnt_size = 35
    label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt_height =  len(character_bytes.split()) * (label_fnt_size + 5)
    txt = Image.new('RGBA', (txt_width, txt_height))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="blue")
    label = txt
    img.paste(label, (350,8))

    # Sprite bytes
    label_text = character_bytes
    label_fnt_size = 35
    label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt_height =  len(character_bytes.split()) * (label_fnt_size + 5)
    txt = Image.new('RGBA', (txt_width, txt_height))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt
    img.paste(label, (470,8))
    
    # Main Sprite image
    #character_image = character_image.resize((character_image.width * 2, character_image.height * 2), Image.NEAREST)
    img.paste(character_image, (30,10))

    return img

def convertCharBitmapToByte(bits):
    bits = bits[::2] # every second element in the list
    bit_string = ''.join(bits)
    byte = ("00" + hex(int(bit_string,2))[2:])[-2:].upper()
    return byte


In [541]:
!mkdir -p text_character_set_diagrams/base_character_set_diagrams

In [542]:
for color_name, color in char_colors.items():
    for character_name in base_character_set:
        character_image = Image.open(f"text_character_set_diagrams/base_character_set_tables/{color_name}_{character_name}.png")
        character_bytes = '\n'.join(["$"+convertCharBitmapToByte(x) for x in base_character_set[character_name]])
        img = generateCharacterDiagram(character_image, character_name, character_bytes)
        img.save(f"text_character_set_diagrams/base_character_set_diagrams/{color_name}_{character_name}.png")
        

### Generate the full character sets

In [543]:
blank_line = "0" * 8
full_character_set = {}
for k,bit_lines in base_character_set.items():
    top_half = [line for lines in zip([blank_line] * 4, bit_lines[:4]) for line in lines]
    full_character_set[k] = top_half
    
    k_int = int(k,16)
    bk = hex(k_int | 0x80)[2:].upper()
    bottom_half = [line for lines in zip([blank_line] * 4, bit_lines[4:]) for line in lines]
    full_character_set[bk] = bottom_half

### Generate PNGs of the full character sets

In [544]:
!mkdir -p text_character_set_diagrams/full_character_sets

In [545]:
for color_name, color in char_colors.items():
    for character_name in full_character_set:
        img = paintRawMonoCharacter(full_character_set, character_name, color=color)
        img = img.resize((img.width * 15, img.height * 15), Image.NEAREST)
        framed_img = img.copy()
        draw = ImageDraw.Draw(framed_img)
        draw.rectangle((0, 0, img.width-1, img.height-1), fill=None, outline="black")

        if character_name:
            img.save(f"text_character_set_diagrams/full_character_sets/{color_name}_{character_name}.png")
            framed_img.save(f"text_character_set_diagrams/full_character_sets/{color_name}_{character_name}_framed.png")


### Generate PNG tables of the full character sets

In [546]:
!mkdir -p text_character_set_diagrams/full_character_set_tables

In [547]:
for color_name, color in char_colors.items():
    for character_name in full_character_set:
        img = paintCharacterDiagram(full_character_set, character_name, color=color)
        if character_name:
            img.save(f"text_character_set_diagrams/full_character_set_tables/{color_name}_{character_name}.png")

### Generate the full character set diagrams

In [548]:
!mkdir -p text_character_set_diagrams/full_character_set_diagrams

In [549]:
for color_name, color in char_colors.items():
    for character_name in full_character_set:
        character_image = Image.open(f"text_character_set_diagrams/full_character_set_tables/{color_name}_{character_name}.png")
        character_bytes = '\n'.join(["$"+convertCharBitmapToByte(x) for x in full_character_set[character_name]])
        img = generateCharacterDiagram(character_image, character_name, character_bytes)
        img.save(f"text_character_set_diagrams/full_character_set_diagrams/{color_name}_{character_name}.png")


### Generate the composite character set diagrams

In [550]:
def generateLeftSideCharacterDiagram(character_image, character_name):

    img = Image.new('RGBA', (360,340))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"CHAR/{character_name}"
    label_fnt_size = 29
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (5, (img.height - (label.height + 10))))

    # Main Sprite image
    img.paste(character_image, (30,10))

    return img

def generateRightSideCharacterDiagram(character_image, character_name):

    img = Image.new('RGBA', (360,340))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"CHAR/{character_name}"
    label_fnt_size = 29
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    label = txt.rotate(90,  expand=1)
    img.paste(label, (character_image.width + 10, (img.height - (label.height + 10))))

    # Main Sprite image
    img.paste(character_image, (5,10))

    return img

In [551]:
!mkdir -p text_character_set_diagrams/full_character_set_diagrams_left
!mkdir -p text_character_set_diagrams/full_character_set_diagrams_right

In [552]:
for color_name, color in char_colors.items():
    for character_name in full_character_set:
        character_image = Image.open(f"text_character_set_diagrams/full_character_set_tables/{color_name}_{character_name}.png")
        img = generateLeftSideCharacterDiagram(character_image, character_name)
        img.save(f"text_character_set_diagrams/full_character_set_diagrams_left/{color_name}_{character_name}.png")
        img = generateRightSideCharacterDiagram(character_image, character_name)
        img.save(f"text_character_set_diagrams/full_character_set_diagrams_right/{color_name}_{character_name}.png")


## Create the 2-character composite diagrams

In [553]:
!mkdir -p text_character_set_diagrams/composite_character_set_diagrams/

In [554]:
color_combos = [("c64_blue", "c64_ltblue"), ("c64_ltblue", "c64_ltgreen"), ("c64_ltred", "c64_blue"), ("c64_red", "c64_ltred"),
                ("c64_ltblue", "c64_blue"), ("c64_white", "c64_yellow"), ("c64_ltgreen", "c64_green")]

for i in list(range(0, 0x3A)) + [0x7C]:
    for top_color_name, bottom_color_name in color_combos:
        top_character_name = f"{i:02X}"
        bottom_character_name = f"{i|0x80:02X}"
        top_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams/{top_color_name}_{top_character_name}.png")
        bottom_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams/{bottom_color_name}_{bottom_character_name}.png")
        bottom_diagram = bottom_diagram.crop((0, 10, bottom_diagram.width, bottom_diagram.height))
        composite_image = Image.new('RGBA', (top_diagram.width, top_diagram.height + bottom_diagram.height - 25))
        composite_image.paste(top_diagram, (0,0))
        composite_image.paste(bottom_diagram, (0, top_diagram.height-19))
        composite_image.save(f"text_character_set_diagrams/composite_character_set_diagrams/{top_color_name}_{bottom_color_name}_{top_character_name}.png")
    

## Create the 4-character composite diagrams

In [555]:
for i in range(0x3A, 0x5a):
    for top_color_name, bottom_color_name in color_combos:
        top_left_character_name = f"{i:02X}"
        bottom_left_character_name = f"{i|0x80:02X}"
        top_right_character_name = f"{i+0x20:02X}"
        bottom_right_character_name = f"{i+0x20|0x80:02X}"

        top_left_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_left/{top_color_name}_{top_left_character_name}.png")
        bottom_left_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_left/{bottom_color_name}_{bottom_left_character_name}.png")
        bottom_left_diagram = bottom_left_diagram.crop((0, 10, bottom_left_diagram.width, bottom_left_diagram.height))

        top_right_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_right/{top_color_name}_{top_right_character_name}.png")
        bottom_right_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_right/{bottom_color_name}_{bottom_right_character_name}.png")
        bottom_right_diagram = bottom_right_diagram.crop((0, 10, bottom_right_diagram.width, bottom_right_diagram.height))

        composite_image = Image.new('RGBA', ((top_left_diagram.width * 2) - 10, (top_left_diagram.height + bottom_left_diagram.height - 10)))
        composite_image.paste(top_left_diagram, (0,0))
        composite_image.paste(bottom_left_diagram, (0, top_left_diagram.height - 6))
        composite_image.paste(top_right_diagram, (top_left_diagram.width - 10, 0))
        composite_image.paste(bottom_right_diagram, (top_left_diagram.width - 10, top_left_diagram.height - 6))
        composite_image.save(f"text_character_set_diagrams/composite_character_set_diagrams/{top_color_name}_{bottom_color_name}_{top_left_character_name}.png")


In [556]:
i = 0x7a
top_left_character_name = f"{i:02X}"
bottom_left_character_name = f"{i|0x80:02X}"
i += 0x01
top_right_character_name = f"{i:02X}"
bottom_right_character_name = f"{i|0x80:02X}"

top_left_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_left/c64_white_{top_left_character_name}.png")
bottom_left_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_left/c64_ltblue_{bottom_left_character_name}.png")
bottom_left_diagram = bottom_left_diagram.crop((0, 10, bottom_left_diagram.width, bottom_left_diagram.height))

top_right_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_right/c64_white_{top_right_character_name}.png")
bottom_right_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_right/c64_ltblue_{bottom_right_character_name}.png")
bottom_right_diagram = bottom_right_diagram.crop((0, 10, bottom_right_diagram.width, bottom_right_diagram.height))

composite_image = Image.new('RGBA', (top_left_diagram.width * 2, (top_left_diagram.height + bottom_left_diagram.height - 10)))
composite_image.paste(top_left_diagram, (0,0))
composite_image.paste(bottom_left_diagram, (0, top_left_diagram.height - 6))
composite_image.paste(top_right_diagram, (top_left_diagram.width - 9, 0))
composite_image.paste(bottom_right_diagram, (top_left_diagram.width - 9, top_left_diagram.height - 6))
composite_image.save(f"text_character_set_diagrams/composite_character_set_diagrams/c64_white_c64_ltblue_{top_left_character_name}.png")


## Create the 2-character composite character sets

In [557]:
!mkdir -p text_character_set_diagrams/composite_character_sets/

In [558]:
for i in list(range(0, 0x3A)) + [0x7C]:
    for top_color_name, bottom_color_name in color_combos:
        top_character_name = f"{i:02X}"
        bottom_character_name = f"{i|0x80:02X}"
        for suffix in ["","_framed"]:
            top_diagram = Image.open(f"text_character_set_diagrams/full_character_sets/{top_color_name}_{top_character_name}{suffix}.png")
            bottom_diagram = Image.open(f"text_character_set_diagrams/full_character_sets/{bottom_color_name}_{bottom_character_name}{suffix}.png")
            bottom_diagram = bottom_diagram.crop((0, 0, bottom_diagram.width, bottom_diagram.height))
            composite_image = Image.new('RGBA', (top_diagram.width, top_diagram.height + bottom_diagram.height))
            composite_image.paste(top_diagram, (0,0))
            composite_image.paste(bottom_diagram, (0, top_diagram.height))
            composite_image.save(f"text_character_set_diagrams/composite_character_sets/{top_color_name}_{bottom_color_name}_{top_character_name}{suffix}.png")

## Create the 4-character composite sets

In [559]:
for i in range(0x3A, 0x5a):
    for top_color_name, bottom_color_name in color_combos:
        top_left_character_name = f"{i:02X}"
        bottom_left_character_name = f"{i|0x80:02X}"
        top_right_character_name = f"{i+0x20:02X}"
        bottom_right_character_name = f"{i+0x20|0x80:02X}"

        for suffix in ["","_framed"]:
            top_left_diagram = Image.open(f"text_character_set_diagrams/full_character_sets/{top_color_name}_{top_left_character_name}{suffix}.png")
            bottom_left_diagram = Image.open(f"text_character_set_diagrams/full_character_sets/{bottom_color_name}_{bottom_left_character_name}{suffix}.png")
            bottom_left_diagram = bottom_left_diagram.crop((0, 0, bottom_left_diagram.width, bottom_left_diagram.height))

            top_right_diagram = Image.open(f"text_character_set_diagrams/full_character_sets/{top_color_name}_{top_right_character_name}{suffix}.png")
            bottom_right_diagram = Image.open(f"text_character_set_diagrams/full_character_sets/{bottom_color_name}_{bottom_right_character_name}{suffix}.png")
            bottom_right_diagram = bottom_right_diagram.crop((0, 0, bottom_right_diagram.width, bottom_right_diagram.height))

            composite_image = Image.new('RGBA', (top_left_diagram.width * 2, (top_left_diagram.height + bottom_left_diagram.height)))
            composite_image.paste(top_left_diagram, (0,0))
            composite_image.paste(bottom_left_diagram, (0, top_left_diagram.height))
            composite_image.paste(top_right_diagram, (top_left_diagram.width, 0))
            composite_image.paste(bottom_right_diagram, (top_left_diagram.width, top_left_diagram.height))
            composite_image.save(f"text_character_set_diagrams/composite_character_sets/{top_color_name}_{bottom_color_name}_{top_left_character_name}{suffix}.png")


In [560]:
i = 0x7a
top_left_character_name = f"{i:02X}"
bottom_left_character_name = f"{i|0x80:02X}"
i += 0x01
top_right_character_name = f"{i:02X}"
bottom_right_character_name = f"{i|0x80:02X}"

top_left_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_left/c64_white_{top_left_character_name}.png")
bottom_left_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_left/c64_ltblue_{bottom_left_character_name}.png")
bottom_left_diagram = bottom_left_diagram.crop((0, 10, bottom_left_diagram.width, bottom_left_diagram.height))

top_right_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_right/c64_white_{top_right_character_name}.png")
bottom_right_diagram = Image.open(f"text_character_set_diagrams/full_character_set_diagrams_right/c64_ltblue_{bottom_right_character_name}.png")
bottom_right_diagram = bottom_right_diagram.crop((0, 10, bottom_right_diagram.width, bottom_right_diagram.height))

composite_image = Image.new('RGBA', (top_left_diagram.width * 2, (top_left_diagram.height + bottom_left_diagram.height - 10)))
composite_image.paste(top_left_diagram, (0,0))
composite_image.paste(bottom_left_diagram, (0, top_left_diagram.height - 6))
composite_image.paste(top_right_diagram, (top_left_diagram.width - 9, 0))
composite_image.paste(bottom_right_diagram, (top_left_diagram.width - 9, top_left_diagram.height - 6))
composite_image.save(f"text_character_set_diagrams/composite_character_sets/c64_white_c64_ltblue_{top_left_character_name}.png")


### Create Filenames with Intuitive Names for Alphanumeric Characters

In [561]:
!mkdir -p text_character_set_diagrams/composite_character_sets_alphanumeric/
!mkdir -p text_character_set_diagrams/full_character_sets_alphanumeric/
!mkdir -p text_character_set_diagrams/base_character_sets_alphanumeric/
!mkdir -p text_character_set_diagrams/composite_character_set_diagrams_alphanumeric/
!mkdir -p text_character_set_diagrams/full_character_set_diagrams_alphanumeric/
!mkdir -p text_character_set_diagrams/base_character_set_diagrams_alphanumeric/

In [562]:
from PIL import Image, ImageFilter, ImageColor

def addBlackOutline(img):
    stroke_radius = 5
    stroke_image = Image.new("RGBA", img.size, (0, 0, 0, 255))
    img_alpha = img.getchannel(3).point(lambda x: 255 if x>0 else 0)
    stroke_alpha = img_alpha.filter(ImageFilter.MaxFilter(stroke_radius))
    # optionally, smooth the result
    stroke_alpha = stroke_alpha.filter(ImageFilter.SMOOTH)
    stroke_image.putalpha(stroke_alpha)
    output = Image.alpha_composite(stroke_image, img)
    return output

In [563]:
import string
lowercase_alphabet = string.ascii_lowercase
uppercase_alphabet = string.ascii_uppercase
numbers = list(range(0,10))
alphanumerics = numbers + list('abcdefghijklInopqrstuv xyz') + list("?!().,:;\"'-=") + [" "] * 10 + list('ABCDEFGHmJKLMNOPQRSTUVWXYZw')

In [564]:
for i in list(range(0, 0x55)):
    literal = alphanumerics[i]
    for top_color_name, bottom_color_name in color_combos:
        character_name = f"{i:02X}"
        for suffix in ["","_framed"]:
            char_image = Image.open(f"text_character_set_diagrams/composite_character_sets/{top_color_name}_{bottom_color_name}_{character_name}{suffix}.png")
            char_image.save(f"text_character_set_diagrams/composite_character_sets_alphanumeric/{top_color_name}_{bottom_color_name}_{literal}{suffix}.png")
            if suffix == "":
                char_image = addBlackOutline(char_image)
                suffix = "_outlined"
                char_image.save(f"text_character_set_diagrams/composite_character_sets_alphanumeric/{top_color_name}_{bottom_color_name}_{literal}{suffix}.png")
                char_image.save(f"text_character_set_diagrams/composite_character_sets/{top_color_name}_{bottom_color_name}_{character_name}{suffix}.png")



In [565]:
for i in list(range(0x55, 0x5a)):
    for top_color_name, bottom_color_name in color_combos:
        character_name = f"{i:02X}"
        char_image = Image.open(f"text_character_set_diagrams/composite_character_sets/{top_color_name}_{bottom_color_name}_{character_name}.png")
        char_image = addBlackOutline(char_image)
        suffix = "_outlined"
        char_image.save(f"text_character_set_diagrams/composite_character_sets/{top_color_name}_{bottom_color_name}_{character_name}_outlined.png")



In [566]:
for i in list(range(0, 0x55)):
    literal = alphanumerics[i]
    for top_color_name, bottom_color_name in color_combos:
        character_name = f"{i:02X}"
        char_image = Image.open(f"text_character_set_diagrams/composite_character_set_diagrams/{top_color_name}_{bottom_color_name}_{character_name}.png")
        char_image.save(f"text_character_set_diagrams/composite_character_set_diagrams_alphanumeric/{top_color_name}_{bottom_color_name}_{literal}.png")

In [567]:
for i in list(range(0, 0x55)):
    literal = alphanumerics[i]
    for color_name, color in char_colors.items():
        character_name = f"{i:02X}"
        for suffix in ["","_framed"]:
            char_image = Image.open(f"text_character_set_diagrams/full_character_sets/{color_name}_{character_name}{suffix}.png")
            char_image.save(f"text_character_set_diagrams/full_character_sets_alphanumeric/{color_name}_{literal}{suffix}.png")
            if suffix == "":
                char_image = addBlackOutline(char_image)
                suffix = "_outlined"
                char_image.save(f"text_character_set_diagrams/full_character_sets_alphanumeric/{top_color_name}_{bottom_color_name}_{literal}{suffix}.png")



In [568]:
for i in list(range(0, 0x55)):
    literal = alphanumerics[i]
    for color_name, color in char_colors.items():
        character_name = f"{i:02X}"
        char_image = Image.open(f"text_character_set_diagrams/full_character_set_diagrams/{color_name}_{character_name}.png")
        char_image.save(f"text_character_set_diagrams/full_character_set_diagrams_alphanumeric/{color_name}_{literal}.png")



In [569]:
for i in list(range(0, 0x55)):
    literal = alphanumerics[i]
    for color_name, color in char_colors.items():
        character_name = f"{i:02X}"
        for suffix in ["","_framed"]:
            char_image = Image.open(f"text_character_set_diagrams/base_character_sets/{color_name}_{character_name}{suffix}.png")
            char_image.save(f"text_character_set_diagrams/base_character_sets_alphanumeric/{color_name}_{literal}{suffix}.png")
            if suffix == "":
                char_image = addBlackOutline(char_image)
                suffix = "_outlined"
                char_image.save(f"text_character_set_diagrams/base_character_sets_alphanumeric/{top_color_name}_{bottom_color_name}_{literal}{suffix}.png")



In [570]:
for i in list(range(0, 0x55)):
    literal = alphanumerics[i]
    for color_name, color in char_colors.items():
        character_name = f"{i:02X}"
        char_image = Image.open(f"text_character_set_diagrams/base_character_set_diagrams/{color_name}_{character_name}.png")
        char_image.save(f"text_character_set_diagrams/base_character_set_diagrams_alphanumeric/{color_name}_{literal}.png")



### Typeset a sentence using the font

In [571]:
char = ""
template = """\\raisebox{-0.1\\totalheight}{\\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_%CHAR%.png}}
\\hspace{-0.2cm}"""

sentence = "The temptation to typeset this chapter entirely in Uridium's font is almost overwhelming. But for your sake, I will resist it."
for char in sentence:
    letter = template.replace("%CHAR%", char)
    print(letter)


\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_T.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_h.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_e.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_ .png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_t.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_e.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_m.

In [572]:
sentence = "It is much easier to read "
for char in sentence:
    letter = template.replace("%CHAR%", char)
    print(letter)


\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_I.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_t.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_ .png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_i.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_s.png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_ .png}}
\hspace{-0.2cm}
\raisebox{-0.1\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_blue_c64_ltblue_m.

In [573]:
#of a Commodore 64 character font that overcomes the limitation
template = """\\raisebox{-0.25\\totalheight}{\\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_%CHAR%.png}}
\\hspace{-0.2cm}"""
sentence = """
there can be no denying the loveliness of a Commodore 64 character
font that overcomes the limitations of its native environment with 
such elan that it can be typeset and read as easily today as it was 
40 years ago. It is as exquisite now as it was when it was first made.
"""
for char in [x for x in sentence if x != '\n']:
    letter = template.replace("%CHAR%", char)
    print(letter)


\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_t.png}}
\hspace{-0.2cm}
\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_h.png}}
\hspace{-0.2cm}
\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_e.png}}
\hspace{-0.2cm}
\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_r.png}}
\hspace{-0.2cm}
\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_e.png}}
\hspace{-0.2cm}
\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_ .png}}
\hspace{-0.2cm}
\raisebox{-0.25\totalheight}{\includegraphics[height=0.45cm]{src/composite_character_sets_alphanumeric/c64_ltred_c64_red_c.png}}
\

## Generate PNGs of the Logo Character Set

In [574]:
!mkdir -p text_character_set_diagrams/logo_character_sets

In [575]:
for color_name, color in char_colors.items():
    for character_name in logo_character_set:
        img = paintRawMonoCharacter(logo_character_set, character_name, color=color)
        img = img.resize((img.width * 15, img.height * 15), Image.NEAREST)
        framed_img = img.copy()
        draw = ImageDraw.Draw(framed_img)
        draw.rectangle((0, 0, img.width-1, img.height-1), fill=None, outline="black")

        if character_name:
            img.save(f"text_character_set_diagrams/logo_character_sets/{color_name}_{character_name}.png")
            framed_img.save(f"text_character_set_diagrams/logo_character_sets/{color_name}_{character_name}_framed.png")


In [517]:
test_img = Image.open(f"text_character_set_diagrams/logo_character_sets/c64_red_0A.png")
char_width, char_height = test_img.width, test_img.height

logo_color_combos = [("c64_red", "c64_ltred")]

for top_color_name, bottom_color_name in logo_color_combos:
    for suffix in ["","_framed"]:
        composite_image = Image.new('RGBA', (char_width * 10, char_height * 2))
        x = 0
        for i in range(0, 0x0A):
            top_character_name = f"{i:02X}"
            bottom_character_name = f"{i+10:02X}"
            top_left_diagram = Image.open(f"text_character_set_diagrams/logo_character_sets/{top_color_name}_{top_character_name}{suffix}.png")
            bottom_left_diagram = Image.open(f"text_character_set_diagrams/logo_character_sets/{bottom_color_name}_{bottom_character_name}{suffix}.png")

            composite_image.paste(top_left_diagram, (x,0))
            composite_image.paste(bottom_left_diagram, (x, top_left_diagram.height))
            x += char_width
        composite_image.save(f"text_character_set_diagrams/logo_character_sets/{top_color_name}_{bottom_color_name}_logo{suffix}.png")


# Scratchpad